# Trading bot: fine-tune Laya on news -> next-hour price move

Runs `scripts/finetune_laya.py` from the trading-bot repo on a Kaggle GPU, once per line of `experiments.txt`
(`<name> <finetune args>`). Inputs come from the private dataset `trading-bot-laya-finetune-data`. Each run writes
`/kaggle/working/<name>/` (model + test_predictions.jsonl) and `/kaggle/working/logs/<name>.log`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Same laya version as the bot, so the checkpoint format matches.
!pip install -q "laya==0.3.10" python-dotenv
import torch, laya
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Rebuild the repo layout the script expects: config.py, advisor/, scripts/, data/.
import glob, os, shutil, subprocess, shlex
files = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*', recursive=True) if os.path.isfile(p)}
print(sorted(files))
repo = '/kaggle/working/repo'
for d in ('advisor', 'scripts', 'data'):
    os.makedirs(f'{repo}/{d}', exist_ok=True)
shutil.copy(files['config.py'], repo)
shutil.copy(files['laya_advisor.py'], f'{repo}/advisor/')
open(f'{repo}/advisor/__init__.py', 'w').close()
shutil.copy(files['finetune_laya.py'], f'{repo}/scripts/')
for f in files:
    if f.endswith('.jsonl'):
        shutil.copy(files[f], f'{repo}/data/')
experiments = [l.split(None, 1) for l in open(files['experiments.txt']) if l.strip() and not l.startswith('#')]
print(experiments)

In [ ]:
os.makedirs('/kaggle/working/logs', exist_ok=True)
for name, extra in experiments:
    cmd = f'python scripts/finetune_laya.py --out /kaggle/working/{name} {extra.strip()}'
    print('=' * 20, name, '|', cmd, flush=True)
    with open(f'/kaggle/working/logs/{name}.log', 'w') as log:
        p = subprocess.Popen(shlex.split(cmd), cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line); log.flush()
            if 'warn' not in line.lower() and 'step' not in line:
                print(line, end='', flush=True)
        p.wait()
    print(name, 'exit code', p.returncode, flush=True)

In [ ]:
# Keep only the models, predictions and logs as output (the repo copy is just inputs).
shutil.rmtree(repo, ignore_errors=True)
!du -sh /kaggle/working/*